# ==================================================
# 🚀 FINAL OPENCODE 3 MODELS SETUP
# ==================================================
# OpenCode with Local LLM on Google Colab
# Models: Qwen3.5-9B, Qwen3-8B, Gemma4-E4B
# ==================================================

In [ ]:
import os, subprocess, time, re, requests, shutil, json

print("=" * 60)
print("🛠️ OPENCODE 3 MODELS SETUP")
print("=" * 60)

## 1. Install System Dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd curl

## 2. Install Ollama (if not already installed)

In [ ]:
if not shutil.which("ollama"):
    print("📥 Installing Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("✅ Ollama already installed")

## 3. Install Cloudflare Tunnel

In [ ]:
if not os.path.exists("cloudflared"):
    print("📥 Installing Cloudflared...")
    !curl -L https://github.com/cloudflare/cloudflare/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
    !chmod +x cloudflared
else:
    print("✅ Cloudflared already installed")

## 4. Start Ollama Service

In [ ]:
!pkill ollama || true
os.environ["OLLAMA_HOST"] = "0.0.0.0"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for Ollama to start
for _ in range(30):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("✅ Ollama is running")
            break
    except:
        time.sleep(1)

## 5. Define the 3 Best Working Models

In [ ]:
MODELS = [
    {"name": "qwen3.5:9b", "new_name": "qwen3.5-9b-32k"},
    {"name": "qwen3:8b", "new_name": "qwen3-8b-32k"},
    {"name": "gemma4:e4b", "new_name": "gemma4-e4b-32k"},
]

## 6. Pull Models and Create 32K Context Versions

In [ ]:
print("\n" + "=" * 60)
print("📥 PULLING & SETTING UP MODELS")
print("=" * 60)

for model in MODELS:
    # Check if model already exists
    check = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    
    if model['name'] not in check.stdout:
        print(f"\n📥 Pulling {model['name']}...")
        try:
            result = subprocess.run(["ollama", "pull", model['name']], capture_output=True, text=True, timeout=300)
            if result.returncode == 0:
                print(f"  ✅ Pulled {model['name']}")
        except Exception as e:
            print(f"  ❌ Error: {e}")
            continue
    else:
        print(f"  ✅ {model['name']} ready")
    
    # Create 32K context version
    check_32k = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    if model['new_name'] in check_32k.stdout:
        print(f"  ✅ {model['new_name']} ready")
    else:
        print(f"  🔧 Creating {model['new_name']} with 32K context...")
        modelfile = f"FROM {model['name']}\nPARAMETER num_ctx 32768\n"
        with open("modelfile.temp", "w") as f:
            f.write(modelfile)
        
        result = subprocess.run(
            ["ollama", "create", "-f", "modelfile.temp", model['new_name']],
            capture_output=True, text=True
        )
        
        if result.returncode == 0:
            print(f"  ✅ Created {model['new_name']}")
        else:
            print(f"  ⚠️ Using default context")

## 7. Show Available Models

In [ ]:
print("\n" + "=" * 60)
print("📋 AVAILABLE MODELS")
print("=" * 60)
list_result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(list_result.stdout)

## 8. Restart Ollama Service

In [ ]:
!pkill ollama || true
time.sleep(2)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

## 9. Start Cloudflare Tunnel & Get Public URL

In [ ]:
print("=" * 60)
print("🌐 OPENING TUNNEL")
print("=" * 60)

!pkill cloudflared || true
p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for _ in range(60):
    line = p.stdout.readline()
    if line:
        print(line.strip())
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group()
        break

if tunnel_url:
    print("\n" + "=" * 60)
    print("✅ READY FOR OPENCODE!")
    print("=" * 60)
    print(f"\n🔗 TUNNEL URL:\n")
    print(f"   {tunnel_url}/v1")
    print(f"\n📋 USE THIS URL IN YOUR CONFIG\n")
    print("📋 AVAILABLE MODELS:")
    print("   1. qwen3.5-9b-32k (DEFAULT - BEST)")
    print("   2. qwen3-8b-32k")
    print("   3. gemma4-e4b-32k")
    print("\n" + "=" * 60)
else:
    print("❌ Tunnel failed")